# Overlap can reveal timing

A three-second word window often contains the start of the next word. Two such windows
share signal samples. A joint decoder can compare their displacement and infer the time
between word onsets, even if the shared signal is random.

We test this on LibriBrain100. First check the geometry. Then train the same joint decoder
with real MEG, shared synthetic signal, and independent synthetic signals. A timing-only
control tests how much the onset intervals alone can predict. No language model is used.


Run the cells from top to bottom. With `RUN = False`, only the small examples run.
Set it to `True` to download data, train, and evaluate. Completed work is reused.

`N_SEEDS = 1` runs one training seed. Increase it to compare independent runs.
Optional experiments are enabled by name in `EXTRA`; they need the models listed in their section.


In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display
from simpleb2t.io import bundled, read, DATA
from simpleb2t.workflows import seed_list, storage_estimate

# Both notebooks share these directories. Change BASE if you keep data on another disk.
BASE = Path.cwd().resolve()
if BASE.name == "notebooks":
    BASE = BASE.parent
RAW = BASE / "raw_data"
WORK = BASE / "runs"
os.environ.setdefault("HF_HOME", str(BASE / ".cache" / "huggingface"))

N_SEEDS = 1
SEEDS = seed_list(N_SEEDS)  # 0, then 100, 200, ...
DEVICE = "cuda"
RUN = False               # True enables the main download/training/evaluation cells.
EXTRA = set()             # Optional experiments are named below.
print(f"Seeds: {SEEDS}; preview: {not RUN}; work directory: {WORK}")


## 1. See the information in the overlap

The two traces are vertically separated for legibility. In the left panel, their shared samples
are identical after shifting by the onset gap. On the right, a new signal is drawn for each
window. Matching samples no longer reveal the gap. The signals and gap below are illustrative.

The recoverable quantity is **onset-to-onset time**. It includes any pause after the first word;
it need not equal the annotated duration of speech.


In [ ]:
from simpleb2t.workflows import overlap_example, overlap_summary
figure, example = overlap_example()
display(pd.Series(example))


How common is this in the actual inputs? Count adjacent eligible words within each natural
sentence. The mean shared fraction below includes zero-overlap pairs; it is not conditioned
on overlap. This calculation uses bundled onset annotations, so no MEG download is needed.


In [ ]:
display(overlap_summary().round(2))


## 2. Inspect the split

Whole recording sessions are assigned to training, validation, or test.
Some passages repeat across sessions, so held-out recordings can contain familiar speech.


In [ ]:
from simpleb2t.data import natural
split_rows = []
for split, metadata in natural().items():
    ids = {item["record"] for item in metadata["items"]}
    split_rows.append(dict(split=split, recordings=len(ids),
                           words=len(metadata["items"]), sentences=len(metadata["sequences"])))
display(pd.DataFrame(split_rows))


## 3. Prepare recordings

PNPL downloads and preprocesses the recordings, then extracts word windows on demand.
It caches each continuous recording once, rather than storing overlapping copies.
Allow 64 GiB RAM and about 200 GiB free disk, including room for checkpoints.

To begin without downloading MEG, keep only `shared_pulses`, `independent_pulses`, and
`timing` in `MODELS`. These controls need only word onsets and small event tables.


In [ ]:
MODELS = ["joint", "single_word", "shared_pulses", "independent_pulses", "timing"]
display(storage_estimate())
from simpleb2t import data
if RUN:
    annotations_only = not any(m in {"joint", "single_word", "ours", "stitched"} for m in MODELS)
    data.download(RAW, annotations_only=annotations_only)
    data.prepare(RAW, WORK, annotations_only=annotations_only)


## 4. Train the controls

All models predict frozen T5 word embeddings with the same inherited D-SigLIP objective.
Training uses the full eligible vocabulary. The 50-word retrieval set is used only for testing.
The best checkpoint is selected on validation, not test.

| Model | Input | What it tests |
|---|---|---|
| `joint` | Real MEG windows, jointly processed | Original contextual decoder |
| `single_word` | Real MEG, Transformer sees one word | Remove comparisons between windows |
| `shared_pulses` | One synthetic field per sentence, cut at real onsets | Preserve overlap without MEG |
| `independent_pulses` | A separate synthetic field for every window | Remove shared-signal overlap |
| `timing` | Log time to next onset | Supply the timing cue directly |

Synthetic fields are drawn afresh during training and are deterministic during evaluation.
They contain neither MEG nor word labels. The independent control still has word order and
sentence length. It need not be exactly at chance. The timing input is standardized with
training statistics; a missing next onset maps to zero after standardization.


In [ ]:
from simpleb2t.training import run as train
if RUN:
    for model in MODELS:
        for seed in SEEDS:
            train(WORK, model, seed, device=DEVICE)


## 5. Evaluate natural test sentences

Each model receives complete eligible natural sentences. We score only positions in the fixed
LibriBrain50 list. Balanced accuracy is the mean recall across word classes. Random uniform
guesses have expected accuracy 2%. There is no averaging of repeated observations and no LM.


In [ ]:
from simpleb2t.evaluation import natural_evaluation
from simpleb2t.workflows import natural_table, summarize
if RUN:
    for model in MODELS:
        for seed in SEEDS:
            natural_evaluation(WORK, model, seed, device=DEVICE)
    results = natural_table(WORK, SEEDS)
    display(summarize(results[results.Condition == "natural"], ["Model"], "Balanced accuracy (%)"))


In [ ]:
if RUN:
    import matplotlib.pyplot as plt
    summary = summarize(results[results.Condition == "natural"], ["Model"], "Balanced accuracy (%)")
    ax = summary["mean"].plot.bar(figsize=(7, 3), color="#2878B5", rot=25)
    if N_SEEDS > 1:
        ax.errorbar(range(len(summary)), summary["mean"], yerr=summary["std"], fmt="none", color="black")
    ax.axhline(2, color="black", ls="--", label="Uniform chance (2%)")
    ax.set(ylabel="Balanced word accuracy (%)", xlabel="")
    ax.legend(frameon=False)
    plt.tight_layout()


If shared synthetic signals outperform independent signals, the decoder can exploit
shared structure without a neural response. The timing-only control tests how useful the
onset intervals are by themselves.


## 6. Optional: remove overlap while preserving sentence words

Add `"stitched"` to `EXTRA` and rerun. The joint decoder is retrained on same-word donors that
never overlap within a sentence. The released training assignments retain 96.3% of eligible
words. We compare original and stitched training, plus isolated CNN + MLP training.

Evaluate all three on the **same retained sentence templates**, once with natural windows
and once with non-overlapping donors. This controls sentence content. The isolated MLP model
here is `ours`; it is different from the single-word Transformer above.


In [ ]:
if "stitched" in EXTRA:
    data.download(RAW)
    data.prepare(RAW, WORK)
    for model in ["joint", "stitched", "ours"]:
        for seed in SEEDS:
            train(WORK, model, seed, device=DEVICE)
            natural_evaluation(WORK, model, seed, device=DEVICE, matched=True)
            natural_evaluation(WORK, model, seed, device=DEVICE, stitched=True)
    matched = natural_table(WORK, SEEDS)
    matched = matched[matched.Condition.isin(["matched_natural", "stitched"])]
    display(summarize(matched, ["Model", "Condition"], "Balanced accuracy (%)"))


## 7. Optional: compare neural and timing predictions

After training `joint`, `shared_pulses`, and `ours`, add `"alignment"` to `EXTRA`.
This fits a word predictor from training onset intervals, then compares its probabilities
with the neural model's probabilities on test. Shuffling the timing predictions gives a baseline.
A larger increase in divergence after shuffling means stronger agreement with timing.


In [ ]:
if "alignment" in EXTRA:
    from simpleb2t.analysis import timing_agreement
    timing_agreement(WORK, device=DEVICE, seeds=SEEDS)
    from simpleb2t.io import read
    display(read(WORK / "analysis" / "timing_agreement.json"))


## 8. Optional: try other LibriBrain100 subjects

Add `"cross_subject"` to `EXTRA` to evaluate the subject-0 joint and shared-synthetic models
on Sherlock1 session 11 in subjects 1–32. This downloads additional recordings.
The story also occurs in subject-0 validation; the people are new, the story is not.


In [ ]:
if "cross_subject" in EXTRA:
    data.download(RAW, cross_subject=True)
    data.prepare(RAW, WORK, cross_subject=True)
    for subject in range(1, 33):
        for model in ["joint", "shared_pulses"]:
            for seed in SEEDS:
                natural_evaluation(WORK, model, seed, device=DEVICE, subject=subject)
    display(natural_table(WORK, SEEDS))


Results are saved in `runs/results/`; checkpoints are in `runs/models/`.
See the [code guide](../docs/CODE_GUIDE.md) if you want to change the models or experiments.
